In [15]:
import torch
import evaluate
import numpy as np
import os
import json

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer, pipeline

In [16]:
dataset = load_dataset("unimelb-nlp/wikiann", "tr")

In [17]:
model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [18]:
label_list = dataset["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Etiketler:", label_list)

Etiketler: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [19]:
label_list = dataset["train"].features["ner_tags"].feature.names

print("=== WIKIANN İLK 20 CÜMLE İNCELEMESİ ===\n")

for i in range(20):
    tokens = dataset["train"][i]["tokens"]
    ner_tags = dataset["train"][i]["ner_tags"]
    
    labels = [label_list[tag] for tag in ner_tags]
    
    print(f"--- Cümle {i+1} ---")
    for token, label in zip(tokens, labels):
        print(f"{token:<20} -> {label}")
    print("\n")

=== WIKIANN İLK 20 CÜMLE İNCELEMESİ ===

--- Cümle 1 ---
3.lük                -> O
maçında              -> O
Slovenya             -> B-ORG
Millî                -> I-ORG
Basketbol            -> I-ORG
Takımı'nı            -> I-ORG
yendikleri           -> O
maçta                -> O
23                   -> O
sayı                 -> O
,                    -> O
6                    -> O
ribaund              -> O
,                    -> O
2                    -> O
blok                 -> O
istatistikleriyle    -> O
oynamış              -> O
ve                   -> O
12                   -> O
faul                 -> O
yaptırmıştır         -> O
.                    -> O


--- Cümle 2 ---
'                    -> O
''                   -> O
Denizlispor          -> B-ORG
''                   -> O
'                    -> O


--- Cümle 3 ---
Hami                 -> B-PER
Mandıralı            -> I-PER
36                   -> O
,                    -> O
Orhan                -> B-PER
Çıkırıkçı        

In [20]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
                
            previous_word_idx = word_idx
            
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [21]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names 
)

print("\n--- Hizalanmış Veri Seti Yapısı ---")
print(tokenized_datasets)


--- Hizalanmış Veri Seti Yapısı ---
DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [22]:
sample = tokenized_datasets["train"][0]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
labels = sample["labels"]

print("\n=== TOKEN VE EŞLEŞEN ETIKET KONTROLÜ ===")
print(f"{'TOKEN':<20} | {'ETİKET ID':<10} | {'ETİKET ADI'}")
print("-" * 50)

for tok, lab in zip(tokens, labels):
    label_name = id2label[lab] if lab != -100 else "MASKE -100"
    print(f"{tok:<20} | {lab:<10} | {label_name}")


=== TOKEN VE EŞLEŞEN ETIKET KONTROLÜ ===
TOKEN                | ETİKET ID  | ETİKET ADI
--------------------------------------------------
[CLS]                | -100       | MASKE -100
3                    | 0          | O
.                    | -100       | MASKE -100
lük                  | -100       | MASKE -100
maçında              | 0          | O
Slovenya             | 3          | B-ORG
Millî                | 4          | I-ORG
Basketbol            | 4          | I-ORG
Takımı               | 4          | I-ORG
'                    | -100       | MASKE -100
nı                   | -100       | MASKE -100
yendi                | 0          | O
##k                  | -100       | MASKE -100
##leri               | -100       | MASKE -100
maçta                | 0          | O
23                   | 0          | O
sayı                 | 0          | O
,                    | 0          | O
6                    | 0          | O
riba                 | 0          | O
##und                

In [23]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("\nModel ve DataCollator eğitime hazır!")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5432.32it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch


Model ve DataCollator eğitime hazır!


In [24]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # -100 etiketlerini (özel token'lar ve subword'ler) değerlendirme dışı bırakıyoruz
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [25]:
print("CUDA Kullanılabilir mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Ekran Kartı Modeli   :", torch.cuda.get_device_name(0))
else:
    print("Şu an CPU modundasın. CUDA sürümünü yüklememiz gerekiyor.")

CUDA Kullanılabilir mi?: True
Ekran Kartı Modeli   : NVIDIA GeForce RTX 5060 Laptop GPU


In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",          
    save_strategy="epoch",          
    learning_rate=2e-5,             
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=3,             
    weight_decay=0.01,
    load_best_model_at_end=True,    
    metric_for_best_model="f1",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [27]:
print("=== EĞİTİM BAŞLIYOR ===")
trainer.train()

print("\n=== TEST SETİ DEĞERLENDİRMESİ ===")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\n--- TEST SONUÇLARI ---")
print(f"Precision : {test_results['eval_precision']:.4f}")
print(f"Recall    : {test_results['eval_recall']:.4f}")
print(f"F1 Score  : {test_results['eval_f1']:.4f}")

=== EĞİTİM BAŞLIYOR ===


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.153141,0.119275,0.895472,0.910113,0.902733,0.965890
2,0.096650,0.114462,0.913221,0.924909,0.919028,0.970525
3,0.067637,0.120760,0.913116,0.927573,0.920288,0.971250


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]



=== TEST SETİ DEĞERLENDİRMESİ ===


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.067637,0.117940,3,0.914448,0.922794,0.918602,0.971874



--- TEST SONUÇLARI ---
Precision : 0.9144
Recall    : 0.9228
F1 Score  : 0.9186


In [28]:
def save_best_model_overall(trainer, tokenizer, test_results, save_dir="./saved_berturk_ner"):
    """
    Modeli yalnızca test setindeki F1 skoru geçmiş rekoru kırarsa kaydeder.
    """
    os.makedirs(save_dir, exist_ok=True)
    metrics_file = os.path.join(save_dir, "best_metrics.json")
    
    current_f1 = test_results.get("eval_f1", 0.0)
    previous_best_f1 = -1.0
    
    # Eğer daha önce bir rekor kaydedilmişse oku
    if os.path.exists(metrics_file):
        try:
            with open(metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)
                previous_best_f1 = data.get("best_f1", -1.0)
        except Exception:
            previous_best_f1 = -1.0

    # Düzeltilen f-string formatı
    prev_f1_display = f"{previous_best_f1:.4f}" if previous_best_f1 != -1.0 else "Yok (İlk Kayıt)"

    print("\n" + "="*45)
    print("     ŞAMPİYON MODEL KAYIT KONTROLÜ     ")
    print("="*45)
    print(f"Mevcut Eğitimin Test F1 Skoru : {current_f1:.4f}")
    print(f"Geçmiş En İyi F1 Skoru        : {prev_f1_display}")

    # Skor Karşılaştırması
    if current_f1 > previous_best_f1:
        print(f"\n🎉 TEBRİKLER! Yeni rekor kırıldı ({current_f1:.4f} > {previous_best_f1}).")
        print(f"Model ve tokenizer '{save_dir}' klasörüne kaydediliyor...")
        
        # 1. En iyi modeli ve tokenizer'ı kaydet
        trainer.save_model(save_dir)
        tokenizer.save_pretrained(save_dir)
        
        # 2. Yeni rekor metriklerini JSON dosyasına yaz
        record_data = {
            "best_f1": current_f1,
            "precision": test_results.get("eval_precision", 0.0),
            "recall": test_results.get("eval_recall", 0.0),
            "accuracy": test_results.get("eval_accuracy", 0.0),
            "full_test_results": test_results
        }
        with open(metrics_file, "w", encoding="utf-8") as f:
            json.dump(record_data, f, ensure_ascii=False, indent=4)
            
        print("✅ Yeni en iyi model ve metrikler başarıyla güncellendi!")
    else:
        print(f"\n⚠️ Mevcut model ({current_f1:.4f}), geçmiş rekoru ({previous_best_f1:.4f}) geçemedi.")
        print("🔒 Eski en iyi model korundu, üzerine yazılmadı.")
    print("="*45)

In [29]:
save_best_model_overall(trainer, tokenizer, test_results, save_dir="./saved_berturk_ner")


     ŞAMPİYON MODEL KAYIT KONTROLÜ     
Mevcut Eğitimin Test F1 Skoru : 0.9186
Geçmiş En İyi F1 Skoru        : 0.9189

⚠️ Mevcut model (0.9186), geçmiş rekoru (0.9189) geçemedi.
🔒 Eski en iyi model korundu, üzerine yazılmadı.


In [30]:
model_path = "./saved_berturk_ner"
print(f"Model '{model_path}' dizininden yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"  
)

manual_sentences = [
    "Ahmet Yılmaz bugün uçakla İzmir'e gitti.",
    "Apple Türkiye, İstanbul'da yeni bir ofis açacağını duyurdu.",
    "Mustafa Kemal Atatürk, 1923 yılında Ankara'da cumhuriyeti ilan etti.",
    "Galatasaray, Şükrü Saracoğlu Stadyumu'nda Fenerbahçe ile karşılaştı.",
    "Boğaziçi Üniversitesi kampüsü, İstanbul Boğazı'nın harika manzarasına sahiptir.",
    "Sağlık Bakanı Fahrettin Koca, yarın saat 14:00'te açıklama yapacak.",
    "Türk Hava Yolları'nın TK2026 sefer sayılı uçağı Berlin'e teker koydu.",
    "Trendyol, yeni operasyon merkezini Kocaeli'nin Gebze ilçesinde açtı.",
    "Prof. Dr. İlber Ortaylı'nın son kitabı Kronik Kitap'tan çıktı.",
    "Elon Musk, SpaceX şirketinin merkezini Teksas'a taşıma kararı aldı.",
    "Tarkan'ın son konseri Harbiye Cemil Topuzlu Açıkhava Tiyatrosu'nda gerçekleşti.",
    "Aselsan mühendisleri, yerli radar sistemini Gölbaşı tesislerinde test etti.",
    "Birleşmiş Milletler Genel Kurulu, New York'ta toplandı.",
    "Togg CEO'su Gürcan Karakaş, Gemlik fabrikasında basına bilgi verdi.",
    "Zeynep, geçen hafta Ankara Üniversitesi'nden mezun oldu.",
    "Hepsiburada, e-ticaret pazarındaki payını artırmak için Londra'da yatırımcılarla buluştu.",
    "ODTÜ Bilgisayar Mühendisliği bölümü öğrencileri Silikon Vadisi'ne gezi düzenledi.",
    "Cengiz Holding, Rize'nin İkizdere ilçesindeki projesine hız verdi.",
    "İstanbul Büyükşehir Belediyesi, Haliç çevresinde yeni park alanları oluşturuyor.",
    "Nobel ödüllü yazar Orhan Pamuk, yeni romanının tanıtımını Paris'te yaptı."
]

output_file = "manual_test_results.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== 20 MANUEL TÜRKÇE CÜMLE NER TAHMİNLERİ ===\n")
    f.write("Model: BERTurk - WikiANN Fine-tuned\n\n")
    
    for idx, sentence in enumerate(manual_sentences, 1):
        predictions = ner_pipeline(sentence)
        
        f.write(f"[{idx}] CÜMLE : {sentence}\n")
        f.write("    TAHMİNLER:\n")
        
        if not predictions:
            f.write("      (Varlık bulunamadı)\n")
        else:
            for entity in predictions:
                entity_group = entity.get('entity_group', entity.get('entity'))
                word = entity['word']
                score = entity['score']
                f.write(f"      • {word:<25} -> {entity_group:<12} (Güven: {score:.2f})\n")
        
        f.write("-" * 60 + "\n")

print(f"✅ İşlem tamamlandı! Sonuçlar '{output_file}' dosyasına kaydedildi.")

Model './saved_berturk_ner' dizininden yükleniyor...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3150.62it/s]
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ İşlem tamamlandı! Sonuçlar 'manual_test_results.txt' dosyasına kaydedildi.
